
# Introduction to Data Search and Access with NASA Earthdata

**OCNG 689 — Satellite Oceanography**

This notebook introduces a practical workflow for **finding, filtering, and accessing NASA Earth science data in Python**.  
We will use three complementary interfaces:

1. **Earthdata Search** (the web user interface)
2. **CMR** (the Common Metadata Repository REST API)
3. **`earthaccess`** (a Python client that wraps search, authentication, and file access)

The notebook uses two ocean-focused examples from **PO.DAAC**:

- **MUR25-JPL-L4-GLOB-v04.2** — GHRSST Level-4 MUR 0.25° global sea surface temperature (SST)
- **SWOT_L2_LR_SSH_EXPERT_D** — SWOT Level-2 low-rate sea surface height, expert product

Why these two examples?

- **MUR25** is a small daily netCDF product (~2 MB per granule), so it is ideal for a first end-to-end example.
- **SWOT** is a modern swath altimetry product and is a good example of **spatially filtered granule search**.

Primary references used in this notebook:

- NASA Earthdata Search: <https://www.earthdata.nasa.gov/data/tools/earthdata-search>
- NASA CMR API: <https://cmr.earthdata.nasa.gov/search/site/docs/search/api.html>
- `earthaccess` docs: <https://earthaccess.readthedocs.io/>
- MUR25 dataset landing page: <https://www.earthdata.nasa.gov/data/catalog/pocloud-mur25-jpl-l4-glob-v04.2-4.2>
- SWOT dataset landing page: <https://podaac.jpl.nasa.gov/dataset/SWOT_L2_LR_SSH_EXPERT_D>



## Learning objectives

By the end of this notebook, you should be able to:

- explain the roles of **Earthdata Search**, **CMR**, **DAACs**, and **Earthdata Login**;
- distinguish a **collection** (dataset) from a **granule** (file);
- search NASA Earthdata programmatically with the **CMR REST API**;
- use **`earthaccess.search_datasets()`** and **`earthaccess.search_data()`**;
- filter results by **time** and **region of interest**;
- authenticate with **Earthdata Login**;
- open or download granules for analysis in **xarray**;
- understand when to use **HTTPS**, **direct S3**, **OPeNDAP**, or **Harmony**-style services.



## Conceptual map of the Earthdata ecosystem

A useful mental model is:

```text
Earthdata Search (web UI)
          ↓
CMR (metadata catalog / API)
          ↓
DAAC archives + services (PO.DAAC, NSIDC, GES DISC, OB.DAAC, ...)
          ↓
Access methods: HTTPS, S3, OPeNDAP, Harmony, cloud-native tools
```

A few key definitions:

- A **collection** is roughly a **dataset**.
- A **granule** is usually a **single file** within that dataset.
- **Earthdata Search** is the user-facing discovery portal.
- **CMR** is the metadata backend that stores collection, granule, variable, tool, and service metadata.
- **Earthdata Login** provides the authentication layer required for many downloads and cloud-access workflows.

In practice:

- Use **Earthdata Search** when you are still exploring.
- Use **CMR** when you want a transparent, reproducible query.
- Use **`earthaccess`** for day-to-day Python workflows.



## Optional package installation

If you are running in a fresh environment, uncomment the next cell.


In [1]:

# Uncomment if needed
# %pip install -q earthaccess requests pandas xarray netCDF4 h5netcdf matplotlib


Note: you may need to restart the kernel to use updated packages.


In [1]:

from pathlib import Path
from pprint import pprint

import matplotlib.pyplot as plt
import pandas as pd
import requests
import xarray as xr

import earthaccess

CMR_COLLECTIONS = "https://cmr.earthdata.nasa.gov/search/collections.json"
CMR_GRANULES = "https://cmr.earthdata.nasa.gov/search/granules.json"


def cmr_search(url: str, **params):
    # Return the list of CMR entries for a JSON search response.
    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()
    payload = response.json()
    return payload["feed"]["entry"]


def maybe_call(obj, attr_name: str):
    # Support both property- and method-style earthaccess APIs.
    value = getattr(obj, attr_name)
    return value() if callable(value) else value


def collection_table(entries):
    rows = []
    for c in entries:
        rows.append(
            {
                "short_name": c.get("short_name"),
                "version": c.get("version_id"),
                "title": c.get("dataset_id"),
                "archive_center": c.get("archive_center"),
                "concept_id": c.get("id"),
                "time_start": c.get("time_start"),
                "time_end": c.get("time_end"),
            }
        )
    return pd.DataFrame(rows)


def first_data_link(entry: dict):
    # Return the first likely data link from a CMR granule entry.
    for link in entry.get("links", []):
        href = link.get("href")
        rel = link.get("rel", "")
        if not href:
            continue
        if link.get("inherited") is True:
            continue
        if "data#" in rel or "download#" in rel:
            return href
    return None


def granule_table(entries):
    rows = []
    for g in entries:
        rows.append(
            {
                "producer_granule_id": g.get("producer_granule_id"),
                "time_start": g.get("time_start"),
                "time_end": g.get("time_end"),
                "updated": g.get("updated"),
                "granule_size_MB": g.get("granule_size"),
                "sample_data_link": first_data_link(g),
            }
        )
    return pd.DataFrame(rows)


def link_or_none(granule, access="external"):
    try:
        links = granule.data_links(access=access)
        return links[0] if links else None
    except Exception:
        return None



## 1) Broad discovery with the CMR REST API

A first step is often an intentionally broad metadata search.

Here we search CMR for collections matching the keyword **"sea surface temperature"** and inspect a few returned collections.


In [2]:

broad_collections = cmr_search(
    CMR_COLLECTIONS,
    keyword="sea surface temperature",
    page_size=10,
)

collection_table(broad_collections)


,short_name,version,title,archive_center,concept_id,time_start,time_end
0,OSTIA-UKMO-L4-GLOB-REP-v2.0,2.0,GHRSST Level 4 OSTIA Global Historical Reproce...,NASA/JPL/PODAAC,C2586786218-POCLOUD,1982-01-01T00:00:00.000Z,2024-01-01T00:00:00.000Z
1,EO:EUM:DAT:SENTINEL-3:SL_2_WST___NRT,2017-07-05,SLSTR Sea Surface Temperatures (SST) in NRT - ...,None,C1588876556-EUMETSAT,2017-07-05T00:00:00.000Z,None
2,EO:EUM:DAT:SENTINEL-3:SL_2_WST___NTC,2017-07-05,SLSTR Sea Surface Temperatures (SST) in NTC - ...,None,C1588876559-EUMETSAT,2017-07-05T00:00:00.000Z,None
3,MUR-JPL-L4-GLOB-v4.1,4.1,GHRSST Level 4 MUR Global Foundation Sea Surfa...,NASA/JPL/PODAAC,C1996881146-POCLOUD,2002-05-31T21:00:00.000Z,None
4,MUR25-JPL-L4-GLOB-v04.2,4.2,GHRSST Level 4 MUR 0.25deg Global Foundation S...,NASA/JPL/PODAAC,C2036880657-POCLOUD,2002-08-31T21:00:00.000Z,None
5,OSTIA-UKMO-L4-GLOB-v2.0,2.0,GHRSST Level 4 OSTIA Global Foundation Sea Sur...,NASA/JPL/PODAAC,C2036877535-POCLOUD,2006-12-31T00:00:00.000Z,None
6,AVHRR_OI-NCEI-L4-GLOB-v2.1,2.1,GHRSST Level 4 AVHRR_OI Global Blended Sea Sur...,NASA/JPL/PODAAC,C2036881712-POCLOUD,2016-01-01T00:00:00.000Z,None
7,CMC0.1deg-CMC-L4-GLOB-v3.0,3.0,GHRSST Level 4 CMC0.1deg Global Foundation Sea...,NASA/JPL/PODAAC,C2036881720-POCLOUD,2016-01-01T00:00:00.000Z,None
8,GAMSSA_28km-ABOM-L4-GLOB-v01,1.0,GHRSST Level 4 GAMSSA_28km Global Foundation S...,NASA/JPL/PODAAC,C2036881735-POCLOUD,2008-07-23T00:00:00.000Z,None
9,G18-ABI-L3C-ACSPO-v2.90,2.90,GHRSST L3C NOAA/ACSPO GOES-18/ABI West America...,NASA/JPL/PODAAC,C2731041317-POCLOUD,2022-06-07T00:00:00.000Z,None



### Notes on CMR search parameters

Common collection-level search parameters include:

- `keyword` — free-text search
- `short_name` — a compact dataset identifier
- `version` or `version_id`
- `provider`
- `temporal`
- `bounding_box`
- `page_size`

For reproducible workflows, **`short_name` + `version`** (or a **concept ID**) is often better than a pure keyword search.



## 2) Target a specific collection: MUR25 SST

Now we move from a broad query to a precise one by using the dataset short name:

- **Short name:** `MUR25-JPL-L4-GLOB-v04.2`

This dataset is a cloud-enabled PO.DAAC collection, stored as netCDF-4, with daily global SST fields on a 0.25° grid.


In [3]:

mur25_collection_entries = cmr_search(
    CMR_COLLECTIONS,
    short_name="MUR25-JPL-L4-GLOB-v04.2",
    page_size=5,
)

mur25_collection = mur25_collection_entries[0]
mur25_collection


{'processing_level_id': '4',
 'cloud_hosted': True,
 'boxes': ['-90 -180 90 180'],
 'has_combine': False,
 'time_start': '2002-08-31T21:00:00.000Z',
 'version_id': '4.2',
 'updated': '2019-08-09T19:47:48.185Z',
 'dataset_id': 'GHRSST Level 4 MUR 0.25deg Global Foundation Sea Surface Temperature Analysis (v4.2)',
 'entry_id': 'MUR25-JPL-L4-GLOB-v04.2_4.2',
 'has_spatial_subsetting': True,
 'has_transforms': False,
 'associations': {'variables': ['V2089830953-POCLOUD',
   'V2089830962-POCLOUD',
   'V2089830949-POCLOUD',
   'V2089830964-POCLOUD',
   'V2089830960-POCLOUD',
   'V2112015416-POCLOUD',
   'V2089830957-POCLOUD',
   'V2089830947-POCLOUD'],
  'services': ['S2164732315-XYZ_PROV', 'S2874702816-XYZ_PROV'],
  'tools': ['TL2245207664-POCLOUD', 'TL2108419875-POCLOUD'],
  'visualizations': ['VIS4086833866-ESDIS',
   'VIS4086833859-ESDIS',
   'VIS4086833856-ESDIS']},
 'has_variables': True,
 'data_center': 'POCLOUD',
 'short_name': 'MUR25-JPL-L4-GLOB-v04.2',
 'organizations': ['NASA/JPL/

In [4]:

pd.Series(
    {
        "short_name": mur25_collection.get("short_name"),
        "version": mur25_collection.get("version_id"),
        "title": mur25_collection.get("dataset_id"),
        "concept_id": mur25_collection.get("id"),
        "archive_center": mur25_collection.get("archive_center"),
        "time_start": mur25_collection.get("time_start"),
        "time_end": mur25_collection.get("time_end"),
    }
)


short_name                                  MUR25-JPL-L4-GLOB-v04.2
version                                                         4.2
title             GHRSST Level 4 MUR 0.25deg Global Foundation S...
concept_id                                      C2036880657-POCLOUD
archive_center                                      NASA/JPL/PODAAC
time_start                                 2002-08-31T21:00:00.000Z
time_end                                                       None
dtype: object


## 3) Granule search with CMR

Once you know the collection, you usually want **granules** (files).

Below, we search for three days of MUR25 granules using the **collection concept ID** plus a **temporal window**.


In [5]:

mur25_granule_entries = cmr_search(
    CMR_GRANULES,
    collection_concept_id=mur25_collection["id"],
    temporal="2024-09-01T00:00:00Z,2024-09-03T23:59:59Z",
    page_size=10,
)

granule_table(mur25_granule_entries)


,producer_granule_id,time_start,time_end,updated,granule_size_MB,sample_data_link
0,None,2024-08-31T21:00:00.000Z,2024-09-01T21:00:00.000Z,2024-09-10T08:15:34.483Z,1.7967691421508794,https://archive.podaac.earthdata.nasa.gov/poda...
1,None,2024-09-01T21:00:00.000Z,2024-09-02T21:00:00.000Z,2024-09-11T08:15:41.207Z,1.8008260726928713,https://archive.podaac.earthdata.nasa.gov/poda...
2,None,2024-09-02T21:00:00.000Z,2024-09-03T21:00:00.000Z,2024-09-12T08:15:37.990Z,1.8046236038208003,https://archive.podaac.earthdata.nasa.gov/poda...
3,None,2024-09-03T21:00:00.000Z,2024-09-04T21:00:00.000Z,2024-09-13T08:16:17.177Z,1.8021383285522463,https://archive.podaac.earthdata.nasa.gov/poda...



### Why use `collection_concept_id`?

It is the most explicit way to tell CMR exactly which dataset you want.

A typical search pattern is:

1. Search **collections** and identify the right dataset.
2. Record its **concept ID**.
3. Search **granules** using that concept ID, then add time and/or region filters.



## 4) Search the same dataset with `earthaccess`

`earthaccess` is often the most convenient interface for Python-based workflows because it handles:

- CMR search,
- Earthdata Login authentication,
- remote opening of files, and
- local downloads.


In [6]:

mur25_dataset = earthaccess.search_datasets(
    short_name="MUR25-JPL-L4-GLOB-v04.2",
    cloud_hosted=True,
)[0]

pprint(mur25_dataset.summary())


{'cloud-info': {'Region': 'us-west-2',
                'S3BucketAndObjectPrefixNames': ['podaac-ops-cumulus-public/MUR25-JPL-L4-GLOB-v04.2/',
                                                 'podaac-ops-cumulus-protected/MUR25-JPL-L4-GLOB-v04.2/'],
                'S3CredentialsAPIDocumentationURL': 'https://archive.podaac.earthdata.nasa.gov/s3credentialsREADME',
                'S3CredentialsAPIEndpoint': 'https://archive.podaac.earthdata.nasa.gov/s3credentials'},
 'concept-id': 'C2036880657-POCLOUD',
 'file-type': "[{'Format': 'netCDF-4', 'FormatType': 'Native', "
              "'FormatDescription': 'netCDF', 'AverageFileSize': 2.0, "
              "'AverageFileSizeUnit': 'MB', "
              "'TotalCollectionFileSizeBeginDate': "
              "'2002-06-01T00:00:00.000Z'}]",
 'get-data': ['https://cmr.earthdata.nasa.gov/virtual-directory/collections/C2036880657-POCLOUD',
              'https://search.earthdata.nasa.gov/search/granules?p=C2036880657-POCLOUD',
              'https://

In [7]:

print("Concept ID:", maybe_call(mur25_dataset, "concept_id"))
print("Version:", maybe_call(mur25_dataset, "version"))
print("Landing page:", maybe_call(mur25_dataset, "landing_page"))

try:
    print("Services:")
    pprint(maybe_call(mur25_dataset, "services"))
except Exception as exc:
    print("Could not list services:", exc)


Concept ID: C2036880657-POCLOUD
Version: 4.2
Landing page: 
Services:
{'S2164732315-XYZ_PROV': [{'meta': {'association-details': {'collections': [{'concept-id': 'C1276812928-GES_DISC'},
                                                                            {'concept-id': 'C2704941577-ORNL_CLOUD'},
                                                                            {'concept-id': 'C1276812875-GES_DISC'},
                                                                            {'concept-id': 'C1276812691-GES_DISC'},
                                                                            {'concept-id': 'C2390408273-ORNL_CLOUD'},
                                                                            {'concept-id': 'C2432584227-ORNL_CLOUD'},
                                                                            {'concept-id': 'C2389176598-ORNL_CLOUD'},
                                                                            {'concept-id': 'C3261062541-ORNL_C


## 5) Authenticate with Earthdata Login

You can usually **search** without logging in, but you generally must **log in before downloading or opening files**.

Recommended credential strategies (in order of convenience):

1. a `~/.netrc` file,
2. environment variables `EARTHDATA_USERNAME` and `EARTHDATA_PASSWORD`,
3. interactive prompt via `earthaccess.login()`.

> Do **not** hard-code passwords into notebooks that you plan to share or publish.


In [ ]:

auth = earthaccess.login()
auth



## 6) Search granules with `earthaccess`

Now we repeat the three-day MUR25 search using `earthaccess.search_data()`.


In [ ]:

mur25_results = earthaccess.search_data(
    short_name="MUR25-JPL-L4-GLOB-v04.2",
    cloud_hosted=True,
    temporal=("2024-09-01", "2024-09-03"),
)

len(mur25_results), mur25_results[0]


In [ ]:

pd.DataFrame(
    {
        "granule": [repr(g) for g in mur25_results],
        "size_MB": [g.size() for g in mur25_results],
        "https_link": [link_or_none(g, access="external") for g in mur25_results],
    }
)



### Access types

For cloud-hosted datasets, `earthaccess` can expose two useful classes of links:

- `access="external"` — HTTPS-style access, appropriate outside the cloud
- `access="direct"` — direct S3 access, most useful when computing **inside AWS us-west-2**

Try both on the first granule:


In [ ]:
print("External HTTPS link:")
print(link_or_none(mur25_results[0], access="external"))

print("\nDirect S3 link (useful in us-west-2):")
print(link_or_none(mur25_results[0], access="direct"))



## 7) Open MUR25 granules directly in Python

Because these granules are small, we can open a few of them directly and concatenate along the time dimension.


In [ ]:

opened = earthaccess.open(mur25_results[:3])

mur25_datasets = [xr.open_dataset(obj) for obj in opened]
mur25_ds = xr.concat(mur25_datasets, dim="time").sortby("time")

mur25_ds


In [ ]:

list(mur25_ds.data_vars)



The dataset landing page lists the main variables in each granule, including:

- `analysed_sst`
- `analysis_error`
- `sst_anomaly`
- `lat`
- `lon`
- `time`

For a quick oceanographic check, we convert the analysed SST from **kelvin to °C** and make a simple global plot.


In [ ]:

sst_c = mur25_ds["analysed_sst"] - 273.15

sst_c.isel(time=0).plot(figsize=(10, 4), robust=True)
plt.title("MUR25 analysed SST (°C) — first selected day")
plt.show()



### Regional subset example: Gulf of Mexico

A common workflow is:

1. use Earthdata / CMR to identify the right collection and granules;
2. open the files in Python;
3. subset in **space**, **time**, and **variables** with `xarray`.


In [ ]:

gulf_of_mexico = sst_c.isel(time=0).sel(lat=slice(18, 32), lon=slice(-98, -80))

gulf_of_mexico.plot(figsize=(8, 4), robust=True)
plt.title("MUR25 SST (°C) over the Gulf of Mexico")
plt.show()



## 8) Optional: download local copies

For small files, local download is often the simplest route.  
For very large collections, direct opening or cloud-native access is usually better.


In [ ]:

download_dir = Path("earthdata_demo_downloads")
download_dir.mkdir(exist_ok=True)

local_files = earthaccess.download(mur25_results[:2], str(download_dir))
local_files



## 9) Spatially filtered swath search: SWOT example

Gridded products such as MUR25 are useful for introducing access patterns, but many satellite oceanography datasets are **swath-based**.

The SWOT Level-2 expert SSH product is a good example. The dataset landing page indicates that it contains sea surface height and related variables on a swath-aligned grid and is distributed as one netCDF-4 file per pass (half orbit).

Here we use `earthaccess.search_data()` with both a **time range** and a **bounding box** over the western North Atlantic / Gulf Stream region.


In [ ]:

swot_dataset = earthaccess.search_datasets(
    short_name="SWOT_L2_LR_SSH_EXPERT_D",
    cloud_hosted=True,
)[0]

pprint(swot_dataset.summary())


In [ ]:

try:
    print("Services available for SWOT dataset:")
    pprint(maybe_call(swot_dataset, "services"))
except Exception as exc:
    print("Could not list services:", exc)


In [ ]:

gulf_stream_bbox = (-80.0, 25.0, -60.0, 45.0)

swot_results = earthaccess.search_data(
    short_name="SWOT_L2_LR_SSH_EXPERT_D",
    cloud_hosted=True,
    temporal=("2024-09-01", "2024-09-30"),
    bounding_box=gulf_stream_bbox,
    count=5,
)

len(swot_results)


In [ ]:

pd.DataFrame(
    {
        "granule": [repr(g) for g in swot_results],
        "size_MB": [g.size() for g in swot_results],
        "https_link": [link_or_none(g, access="external") for g in swot_results],
        "s3_link": [link_or_none(g, access="direct") for g in swot_results],
    }
)



This SWOT example highlights an important distinction:

- for a **global grid**, spatial filtering often happens *after* the file is opened;
- for a **swath product**, a spatial query can reduce the list of candidate granules before you open anything.



## 10) Which interface should you use?

| Task | Best first tool |
|---|---|
| I do not know the dataset name yet | **Earthdata Search** or `earthaccess.search_datasets()` |
| I want a fully scripted, transparent metadata query | **CMR REST API** |
| I want to authenticate, download, or open files in Python | **`earthaccess`** |
| I am working in AWS us-west-2 and want cloud-native access | `earthaccess` + **direct S3 links** |
| I need server-side subsetting / transformation | collection **services** such as **Harmony** or **OPeNDAP** (when available) |

A practical rule:

- **Explore broadly with Earthdata Search**
- **Make the search reproducible with CMR or `earthaccess`**
- **Analyze with xarray, pandas, and your preferred scientific Python tools**



## 11) Exercises

1. Change the MUR25 date range from three days to one month. How many granules are returned?
2. Replace the Gulf of Mexico subset with the **California Current** or **equatorial Pacific**.
3. Search for SWOT granules over another dynamic region (for example, the Agulhas retroflection or the Kuroshio Extension).
4. Compare a **broad keyword search** with a **short-name search**. Which is more reproducible?
5. Plot `sst_anomaly` instead of `analysed_sst`.
6. Use the data link from a CMR granule search and confirm that it matches an `earthaccess` result.



## 12) Troubleshooting notes

- **No search results?** Start broad (`keyword` or `short_name` only), then add filters one at a time.
- **Authentication issues?** Confirm your Earthdata Login works and that your credentials are available via `.netrc`, environment variables, or interactive prompt.
- **Remote open fails?** Try downloading the file first, especially for small products.
- **Large files?** Prefer spatial/temporal filtering, cloud-native access, OPeNDAP, or Harmony-like services when available.
- **Protected data or special access controls?** Some collections may require additional user agreements or profile setup.



## References and further reading

### Core system references

- Earthdata Search: <https://www.earthdata.nasa.gov/data/tools/earthdata-search>
- NASA CMR overview: <https://www.earthdata.nasa.gov/about/esdis/eosdis/cmr>
- NASA CMR API docs: <https://cmr.earthdata.nasa.gov/search/site/docs/search/api.html>
- Earthdata Login tutorial: <https://www.earthdata.nasa.gov/learn/tutorials/access-nasa-earth-science-data-earthdata-login>
- `earthaccess` quick start: <https://earthaccess.readthedocs.io/en/stable/quick-start/>
- `earthaccess` search guide: <https://earthaccess.readthedocs.io/en/stable/user_guide/search/>
- `earthaccess` granule results: <https://earthaccess.readthedocs.io/en/stable/user-reference/granules/granules/>

### Ocean data examples

- MUR25 dataset landing page: <https://www.earthdata.nasa.gov/data/catalog/pocloud-mur25-jpl-l4-glob-v04.2-4.2>
- SWOT expert SSH landing page: <https://podaac.jpl.nasa.gov/dataset/SWOT_L2_LR_SSH_EXPERT_D>
- PO.DAAC cookbook example for Python cloud access: <https://podaac.github.io/tutorials/external/access-cloud-python.html>

### Scientific context

- Chin, T. M., Vazquez-Cuervo, J., & Armstrong, E. M. (2017). *A multi-scale high-resolution analysis of global sea surface temperature*. **Remote Sensing of Environment, 200**, 154–169. <https://doi.org/10.1016/j.rse.2017.07.029>
